# FitResultReference

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `analysis`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/FitResultReference.md`


Notebook source link: [FitResultReference.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/FitResultReference.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "FitResultReference"
FAMILY = "analysis"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")


In [ ]:
# Analysis/Fit workflow: build a Poisson GLM with two covariates.
time = np.linspace(0.0, 6.0, 6001)
dt = float(time[1] - time[0])
stim_1 = np.sin(2.0 * np.pi * 0.8 * time)
stim_2 = np.cos(2.0 * np.pi * 0.35 * time + 0.25)
X = np.column_stack([stim_1, stim_2])

true_model = CIFModel(coefficients=np.array([0.45, -0.25]), intercept=np.log(8.0), link="poisson")
true_rate = true_model.evaluate(X)
spike_times = true_model.simulate_by_thinning(time, X, rng=rng)

cov_1 = Covariate(time=time, data=stim_1, name="stim_1", labels=["stim_1"])
cov_2 = Covariate(time=time, data=stim_2, name="stim_2", labels=["stim_2"])
spikes = SpikeTrain(spike_times=spike_times, t_start=float(time[0]), t_end=float(time[-1]), name="unit_1")
trial = Trial(spikes=SpikeTrainCollection([spikes]), covariates=CovariateCollection([cov_1, cov_2]))
config = TrialConfig(covariate_labels=["stim_1", "stim_2"], sample_rate_hz=1.0 / dt, fit_type="poisson")
fit = Analysis.fit_trial(trial, config)
est_rate = fit.predict(X)

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
axes[0].plot(time, stim_1, label="stim_1", linewidth=1.0)
axes[0].plot(time, stim_2, label="stim_2", linewidth=1.0)
axes[0].set_title(f"{TOPIC}: inputs")
axes[0].legend(loc="upper right")

axes[1].plot(time, true_rate, label="true rate", linewidth=1.2)
axes[1].plot(time, est_rate, label="estimated rate", linewidth=1.1)
axes[1].set_ylabel("Hz")
axes[1].legend(loc="upper right")

centers, counts = spikes.bin_counts(bin_size_s=0.02)
axes[2].bar(centers, counts, width=0.018, color="tab:gray")
axes[2].set_xlabel("time [s]")
axes[2].set_ylabel("count/bin")

plt.tight_layout()
plt.show()

coef_error = float(np.linalg.norm(fit.coefficients - np.array([0.45, -0.25])))
print("AIC", float(fit.aic()), "BIC", float(fit.bic()), "coef_error", coef_error)
assert np.isfinite(coef_error)
assert coef_error < 1.5, "Coefficient fit drifted too far from simulation truth"


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
